# Biomedical Image Segmentation Workflow

This notebook demonstrates the same workflow implemented in `src/run_pipeline.py`: load an image, preprocess it, segment objects, quantify regions, and export measurements.


In [ ]:
from pathlib import Path
import sys
sys.path.append('../src')

from preprocess import load_image, crop_image, extract_channel, denoise_median, apply_gamma, threshold_image
from segment import watershed_segmentation
from quantify import region_properties_to_dataframe, save_measurements
from visualize import show_image, plot_segmentation_overlay, plot_area_histogram


## 1. Load image

Place an example image in `data/` and update the path below. Do not commit restricted or proprietary images to GitHub.


In [ ]:
image_path = Path('../data/example_image.png')
image = load_image(image_path)
show_image(image, title='Input image')


## 2. Preprocess image


In [ ]:
# Optional crop: image = crop_image(image, (580, 4100, 1900, 5500))
channel_img = extract_channel(image, channel=0)
denoised = denoise_median(channel_img, size=2)
gamma_corrected = apply_gamma(denoised, gamma=5)
binary_mask = threshold_image(gamma_corrected, threshold=0.65, invert=True)
show_image(binary_mask, title='Thresholded foreground mask')


## 3. Segment features with watershed


In [ ]:
labels = watershed_segmentation(binary_mask)
plot_segmentation_overlay(channel_img, labels)


## 4. Quantify segmented regions


In [ ]:
df = region_properties_to_dataframe(labels, intensity_image=channel_img)
df.head()


In [ ]:
save_measurements(df, '../outputs/region_measurements.csv')
plot_area_histogram(df)
